# 01 Feature Extraction

## Setup and imports

In [1]:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

In [2]:
requirements_path = "/content/drive/MyDrive/xai-project5/requirements.txt" if IN_COLAB else "./../../requirements.txt"

!pip install -r "{requirements_path}"

In [3]:
import numpy as np
import torch
import torch.nn.functional as F
import torch.nn as nn
import open_clip
from PIL import Image
from datasets import load_dataset
from tqdm import tqdm # For the progress bar
import os
import io

In [4]:
if IN_COLAB:
    save_dir = '/content/drive/MyDrive/xai-project5/results/01_feature_extraction'
else:
    save_dir = os.path.abspath(os.path.join('..', 'results', '01_feature_extraction'))

os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, 'chest_embeddings_newModel.pt')
print(f"Save configured at: {save_path}")

Save configured at: /Users/riccardo/Library/CloudStorage/GoogleDrive-riccardomarconi01@gmail.com/Il mio Drive/Progetti/04 - Explainable and Trustworthy AI/Project/xai-project5/src/results/01_feature_extraction/chest_embeddings_newModel.pt


## Model

In this section, we use the **BioMedCLIP** (`microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224`) to extract visual embeddings from medical images.

BioMedCLIP is a state-of-the-art multimodal foundation model, pre-trained on PMC-15M (a large-scale dataset containing 15 million image-text pairs extracted from PubMed Central). Compared to previous models, it provides significantly richer and more robust latent representations, making it ideal for the unsupervised discovery framework. This model provides:

- **Vision Encoder (ViT-B/16)**: Processes medical images (such as chest X-rays) and transforms them into dense vector embeddings (dimension 512).
- **Text Encoder (PubMedBERT)**: Replaces the standard text encoder with a domain-specific language model for the biomedical field, extending the context window up to 256 tokens to capture complex clinical reports and descriptions.
- **Multimodal Alignment**: Visual and textual embeddings are projected into the same shared latent space. This allows for extremely precise cosine similarity computation, a fundamental step for the subsequent semantic alignment of the concepts discovered by the Sparse Autoencoder.

In [5]:
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

model_id = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"

# Instantiate model, image transforms, and tokenizer via open_clip
model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(model_id)
tokenizer = open_clip.get_tokenizer(model_id)

model = model.to(device)
model.eval() # it is already pre-trained

CustomTextCLIP(
  (visual): TimmModel(
    (trunk): VisionTransformer(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
        (norm): Identity()
      )
      (pos_drop): Dropout(p=0.0, inplace=False)
      (patch_drop): Identity()
      (norm_pre): Identity()
      (blocks): Sequential(
        (0): Block(
          (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
          (attn): Attention(
            (qkv): Linear(in_features=768, out_features=2304, bias=True)
            (q_norm): Identity()
            (k_norm): Identity()
            (attn_drop): Dropout(p=0.0, inplace=False)
            (norm): Identity()
            (proj): Linear(in_features=768, out_features=768, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
          )
          (ls1): Identity()
          (drop_path1): Identity()
          (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
          

## Dataset

To successfully implement the MedConcept framework and avoid SAE collapse due to limited data, we decouple our training and evaluation datasets:
 
**1. NIH Chest X-ray 10k Subset - For SAE Training**
- **Content:** A 10,000 image subset of the massive dataset collected by the National Institutes of Health Clinical Center. It provides images with basic disease labels but *lacks* free-text radiology reports.
- **Motivation:** Dictionary learning (SAE) requires a visually diverse set of embeddings to avoid "dead neurons" and successfully disentangle polysemantic features. The NIH 10k dataset provides a good scale and variance for the training phase while being lightweight, fast to download, and fully supported by modern Hugging Face libraries. 
 
**2. Open-I (Indiana University) - For Final Evaluation**
- **Content:** A smaller dataset containing chest X-ray images paired with their corresponding diagnostic reports, collected by Indiana University. Each radiology report includes four sections:
   - The **Comparison** section contains prior patient information (e.g., previous medical examinations).
   - The **Indication** section details symptoms (e.g., hypoxia) or reasons for the examination (e.g., age).
   - The **Findings** section lists radiological observations.
   - The **Impression** section outlines the final diagnosis.
- **Motivation:** Used exclusively to evaluate the frozen SAE. To validate that the discovered concepts are clinically meaningful, we need textual ground truth. The detailed "Findings" and "Impression" sections allow us to pass patient-level concept summaries to an external LLM (e.g., MedGemma) to quantitatively calculate our Aligned, Unaligned, and Uncertain scores.


To ensure semantic correctness and align one image at a time with the ground truth, we only use img_frontal.

The extracted embeddings are **L2 normalized**, ensuring that each vector has unit norm. This is fundamental to:
1. Stabilize the Sparse Autoencoder training
2. Simplify the cosine similarity computation (which becomes a simple matrix multiplication)
3. Semantically interpret the latent neurons of the SAE

### Extraction for SAE training (NIH Chest X-ray 10k Subset)

In [8]:
try:
    for split_name in ["train", "test"]:
        print(f"Downloading NIH Chest X-ray dataset ({split_name})...")
        dataset_nih = load_dataset("g-ronimo/NIH-Chest-X-ray-dataset_10k", split=split_name)

        nih_embeddings = []
        print(f"Extracting visual embeddings from NIH ({split_name} - {len(dataset_nih)} images)....")

        with torch.no_grad():
            for item in tqdm(dataset_nih):
                try:
                    image = item['image'].convert("RGB")
                    
                    image_tensor = preprocess_val(image).unsqueeze(0).to(device)
                    vision_features = model.encode_image(image_tensor)
                    
                    # L2 Normalization
                    vision_embeddings = F.normalize(vision_features, p=2, dim=1)
                    
                    nih_embeddings.append(vision_embeddings.cpu())

                except Exception as e:
                    continue

        if nih_embeddings:
            final_nih_embeddings = torch.cat(nih_embeddings, dim=0)
            
            # We only save the tensor, reports are not needed for SAE training
            nih_save_path = os.path.join(save_dir, f"biomedclip_nih_embeddings_{split_name}.pt")
            torch.save(final_nih_embeddings, nih_save_path)
            print(f"\nNIH save completed: {nih_save_path}")
            print(f"Shape of the training tensor for the SAE: {final_nih_embeddings.shape}\n")
            
            # Free up RAM
            del nih_embeddings
            del final_nih_embeddings
        
        del dataset_nih
        
except Exception as e:
    print(f"Error loading the NIH dataset: {e}\nMake sure you have enough disk space.")

Extracting visual embeddings from NIH (train - 7500 images)....


100%|██████████| 7500/7500 [03:28<00:00, 36.01it/s]



NIH save completed: /Users/riccardo/Library/CloudStorage/GoogleDrive-riccardomarconi01@gmail.com/Il mio Drive/Progetti/04 - Explainable and Trustworthy AI/Project/xai-project5/src/results/01_feature_extraction/biomedclip_nih_embeddings_train.pt
Shape of the training tensor for the SAE: torch.Size([7500, 512])

Extracting visual embeddings from NIH (test - 2500 images)....


100%|██████████| 2500/2500 [00:56<00:00, 43.90it/s]


NIH save completed: /Users/riccardo/Library/CloudStorage/GoogleDrive-riccardomarconi01@gmail.com/Il mio Drive/Progetti/04 - Explainable and Trustworthy AI/Project/xai-project5/src/results/01_feature_extraction/biomedclip_nih_embeddings_test.pt
Shape of the training tensor for the SAE: torch.Size([2500, 512])



### Extraction for evaluation (Open-I Dataset)

In [6]:
print("Downloading the Open-I dataset...")
dataset_openi = load_dataset("ykumards/open-i", split="train")

openi_embeddings = []
openi_reports = []

print("Extracting visual embeddings....")

with torch.no_grad():
    for item in tqdm(dataset_openi):
        try:
            image_bytes = item['img_frontal'] #img_frontal and img_lateral are raw bytes

            if image_bytes is None or len(image_bytes) == 0:
                print(f"Sample with ID {item.get('uid')} has an empty image!")
                continue
            if not image_bytes or not isinstance(image_bytes, bytes):
                continue
            
            image = Image.open(io.BytesIO(image_bytes)).convert("RGB") #We use BytesIO to manage this bytes

            image_tensor = preprocess_val(image).unsqueeze(0).to(device)
            vision_features = model.encode_image(image_tensor) # dim: (batch_size, embedding_dimension)
            
            # L2 Normalization
            vision_embeddings = F.normalize(vision_features, p=2, dim=1)
            
            # Building the ground truth
            findings = item.get('findings', '')
            impression = item.get('impression', '')
            
            if findings is None: findings = ""
            if impression is None: impression = ""
            
            full_report = f"{findings} {impression}".strip()
        
            if len(full_report) < 10:
                continue
                
            openi_embeddings.append(vision_embeddings.cpu())
            openi_reports.append(full_report)

        except Exception as e:
            print(f"Error during sample processing: {e}")
            continue

if openi_embeddings:
    final_openi_embeddings = torch.cat(openi_embeddings, dim=0)
    
    dataset_dict = {
        "embeddings": final_openi_embeddings,
        "reports": openi_reports 
    }
    
    # Creation of directories and saving the results
    openi_save_path = os.path.join(save_dir, "biomedclip_openi_embeddings_with_reports.pt")
    torch.save(dataset_dict, openi_save_path)
    print(f"\nOpen-I save completed: {openi_save_path}")
    print(f"Total samples for final evaluation: {len(openi_reports)}")
    print(f"Shape of the evaluation tensor: {final_openi_embeddings.shape}")
else:
    print("\nNo valid data extracted. Check the dataset loading.")

Extracting visual embeddings....


  2%|▏         | 79/3851 [00:03<02:08, 29.37it/s]

Sample with ID 74 has an empty image!


  2%|▏         | 83/3851 [00:03<01:59, 31.46it/s]

Sample with ID 81 has an empty image!


...

Sample with ID 3950 has an empty image!


 99%|█████████▉| 3820/3851 [02:17<00:01, 27.75it/s]

Sample with ID 3963 has an empty image!


100%|██████████| 3851/3851 [02:18<00:00, 27.74it/s]


Open-I save completed: /Users/riccardo/Library/CloudStorage/GoogleDrive-riccardomarconi01@gmail.com/Il mio Drive/Progetti/04 - Explainable and Trustworthy AI/Project/xai-project5/src/results/01_feature_extraction/biomedclip_openi_embeddings_with_reports.pt
Total samples for final evaluation: 3666
Shape of the evaluation tensor: torch.Size([3666, 512])


In [7]:
embeddings = dataset_dict["embeddings"]
reports = dataset_dict["reports"]

print("--- Inspecting the first 2 elements ---")
for i in range(2):
    print(f"Item {i+1}:")
    
    # Print the text report
    print(f"  Report: {reports[i]}")
    
    # Print info about the corresponding embedding
    emb = embeddings[i]
    if hasattr(emb, 'shape'):
        print(f"  Embedding Shape: {emb.shape}")
    
    # Print just the first 5 values
    print(f"  Embedding (first 5 values): {emb[:5]}")


--- Inspecting the first 2 elements ---
Item 1:
  Report: The cardiac silhouette and mediastinum size are within normal limits. There is no pulmonary edema. There is no focal consolidation. There are no XXXX of a pleural effusion. There is no evidence of pneumothorax. Normal chest x-XXXX.
  Embedding Shape: torch.Size([512])
  Embedding (first 5 values): tensor([-0.0746, -0.0400, -0.2510, -0.0141, -0.0297])
Item 2:
  Report: Borderline cardiomegaly. Midline sternotomy XXXX. Enlarged pulmonary arteries. Clear lungs. Inferior XXXX XXXX XXXX. No acute pulmonary findings.
  Embedding Shape: torch.Size([512])
  Embedding (first 5 values): tensor([-0.0494, -0.0461, -0.2732, -0.0070, -0.0148])
